In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


from matplotlib.patches import Patch
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import os
from sklearn.decomposition import PCA

from sklearn.metrics import f1_score

from sklearn.linear_model import LinearRegression

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC

In [47]:
#Get data
PATHX = "DD//X_TR.csv"
PATHY = "DD//Y_TR.csv"

X = pd.read_csv(PATHX, sep = ",", index_col =0)
labels = pd.read_csv(PATHY, header=0, sep = "/n", index_col=False)

#Checking what labels we have
existing_labels = []
for i in labels["class"]:
    if i not in existing_labels:
        existing_labels.append(i)

print(existing_labels)

x = np.array(X)
labels = np.array(labels).ravel()
print(x.shape)

#scaler = StandardScaler().fit(X)
#X_scaled = scaler.transform(X)
print(labels.shape)


[2, 5, 6, 4, 1, 7, 3]
(8080, 819)
(8080,)


C:\Users\Min Dator\AppData\Local\Temp\ipykernel_41440\4272421157.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  labels = pd.read_csv(PATHY, header=0, sep = "/n", index_col=False)


In [39]:
#Functions and values 

p_val = 10
b = 10
#Calculate the SVD and split the data into train, val and test


def split_data(X, y, random_seed, n, val_fraction = 0.2):
    pca = PCA(n_components=n)
    X_pca = pca.fit_transform(X)

    X_train, X_valid, y_train, y_valid = train_test_split(X_pca, y, test_size=int(len(X)*val_fraction))

    return X_train, y_train, X_valid, y_valid


In [ ]:
components = np.arange(1,100, 2)
c_values = [0.1, 0.5, 1.0, 1.5]
num_trees = [25, 50, 100]
sample_sizes = [0.2, 0.5, 0.8, 1]

models_tested = ["LinearSVC", "Logistic", "Random forest"]
possible_values = {"LinearSVC": {"pca__n_components": components}, "Logistic": {"Classifier__C": c_values,"pca__n_components": components}, "RF": {"Classifier__n_estimators": num_trees, "pca__n_components": components}}
results = [{"best_parameters": [], "f1_score": []}, {"best_parameters": [], "f1_score": []}, {"best_parameters": [], "f1_score": []}]


def training(random_seed, results, possible_values, X, y, size):
    X, _, y, _ = train_test_split(X, y, test_size = size, random_state=random_seed)
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y, 
        test_size=0.2,
        stratify=y,
        random_state=random_seed
    )

    #Linear regression 
    pipe_linear = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('Classifier', LinearSVC())
    ])
    #find optimal values for linear
    opt_linear =RandomizedSearchCV(
        pipe_linear,
        param_distributions=possible_values["LinearSVC"],
        n_iter = 10,
        cv = StratifiedKFold(n_splits=3, shuffle = True, random_state = random_seed),
        scoring = "f1_macro",
    )

    opt_linear.fit(X_train, y_train)
    y_pred_linear = opt_linear.predict(X_valid)
    y_pred_linear = y_pred_linear

    print(y_pred_linear)
    
    f1_linear = f1_score(y_valid, y_pred_linear, average = "macro")
    print(f"Best estimator for linear is {opt_linear.best_estimator_} with score {opt_linear.best_score_} and parameters {opt_linear.best_params_}")
    print(f"f1 score: {f1_linear}")
    results[0]["best_parameters"].append(opt_linear.best_params_)
    results[0]["f1_score"].append(f1_linear)



    pipe_logistic = Pipeline([
        ('scaler', StandardScaler()),   
        ('pca', PCA()),
        ('Classifier', LogisticRegression())
    ])
    #find optimal values for linear
    opt_logistic =RandomizedSearchCV(
        pipe_logistic,
        param_distributions=possible_values["Logistic"],
        n_iter = 10,
        cv = StratifiedKFold(n_splits=3, shuffle = True, random_state = random_seed),
        scoring = "f1_macro",
    )

    opt_logistic.fit(X_train, y_train)
    y_pred_logistic = opt_logistic.predict(X_valid)
    f1_logistic = f1_score(y_valid, y_pred_logistic, average = "macro")
    print(f"Best estimator for logistic is {opt_logistic.best_estimator_} with score {opt_logistic.best_score_} and parameters {opt_logistic.best_params_}")
    print(f"f1 score: {f1_logistic}")
    results[1]["best_parameters"].append(opt_logistic.best_params_)
    results[1]["f1_score"].append(f1_logistic)



    pipe_rf = Pipeline([
        ('scaler', StandardScaler()),   
        ('pca', PCA()),
        ('Classifier', RandomForestClassifier())
    ])
    #find optimal values for linear
    opt_rf =RandomizedSearchCV(
        pipe_rf,
        param_distributions=possible_values["RF"],
        n_iter = 10,
        cv = StratifiedKFold(n_splits=3, shuffle = True, random_state = random_seed),
        scoring = "f1_macro",
    )

    opt_rf.fit(X_train, y_train)
    y_pred_rf = opt_rf.predict(X_valid)
    f1_rf = f1_score(y_valid, y_pred_rf, average = "macro")
    print(f"Best estimator for RF is {opt_rf.best_estimator_} with score {opt_rf.best_score_} and parameters {opt_rf.best_params_}")
    print(f"f1 score: {f1_rf}")
    results[2]["best_parameters"].append(opt_rf.best_params_)
    results[2]["f1_score"].append(f1_rf)


    return results

for
results = training(123, results, possible_values, X, labels, sample_size)   

"""
x_train_proj, y_train, x_valid_proj, y_valid = split_data(X, labels, 10)

lda = LinearDiscriminantAnalysis()
lda.fit(x_train_proj, y_train)

pred_valid_lda = lda.predict(x_valid_proj)
err_valid_linear = np.sum(pred_valid_lda != y_valid)/len(y_valid)


print(err_valid_linear)

cm = confusion_matrix(y_valid, pred_valid_lda)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
"""

[1 7 4 ... 2 4 5]
Best estimator for linear is Pipeline(steps=[('scaler', StandardScaler()),
                ('pca', PCA(n_components=np.int64(81))),
                ('Classifier', LinearSVC())]) with score 0.7526082707285856 and parameters {'pca__n_components': np.int64(81)}
f1 score: 0.7694511817843959
Best estimator for logistic is Pipeline(steps=[('scaler', StandardScaler()),
                ('pca', PCA(n_components=np.int64(97))),
                ('Classifier', LogisticRegression(C=0.1))]) with score 0.8008463473946317 and parameters {'pca__n_components': np.int64(97), 'Classifier__C': 0.1}
f1 score: 0.8194650966200447
Best estimator for RF is Pipeline(steps=[('scaler', StandardScaler()),
                ('pca', PCA(n_components=np.int64(43))),
                ('Classifier', RandomForestClassifier())]) with score 0.7930990628584306 and parameters {'pca__n_components': np.int64(43), 'Classifier__n_estimators': 100}
f1 score: 0.8359080958794822


"\nx_train_proj, y_train, x_valid_proj, y_valid = split_data(X, labels, 10)\n\nlda = LinearDiscriminantAnalysis()\nlda.fit(x_train_proj, y_train)\n\npred_valid_lda = lda.predict(x_valid_proj)\nerr_valid_linear = np.sum(pred_valid_lda != y_valid)/len(y_valid)\n\n\nprint(err_valid_linear)\n\ncm = confusion_matrix(y_valid, pred_valid_lda)\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues')\nplt.xlabel('Predicted')\nplt.ylabel('Actual')\nplt.title('Confusion Matrix')\nplt.show()\n"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name in part1_summary["Model"].unique():
    model_df = (
        part1_summary[part1_summary["Model"] == model_name]
        .sort_values("Fraction")
    )

    axes[0].plot(
        model_df["Fraction"],
        model_df["MeanBrier"],
        marker="o",
        label=model_name
    )
    axes[1].plot(
        model_df["Fraction"],
        model_df["MeanLogLoss"],
        marker="o",
        label=model_name
    )

axes[0].set_xlabel("Sample fraction")
axes[0].set_ylabel("Mean Brier score")
axes[0].set_title("Part 1: Brier score by sample size")
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Sample fraction")
axes[1].set_ylabel("Mean log-loss")
axes[1].set_title("Part 1: Log-loss by sample size")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [10]:
#print(X.min()) 
#print("HUH")
#print(X.max())
#print("x max above")
print(X.describe().T.head())
print("X describe above")
import numpy as np

corr = np.corrcoef(X.values.T)
print(corr.shape)
print("Corr above")

    count      mean       std       min       25%       50%       75%  \
1  8080.0 -0.196407  2.648489 -6.197565 -2.261425 -0.527415  1.592917   
2  8080.0  0.937874  3.135852 -6.411426 -1.385900  0.609143  2.884129   
3  8080.0 -0.345462  2.978839 -6.674499 -2.837162 -0.731263  1.937492   
4  8080.0  0.201644  2.569414 -6.074230 -1.578429 -0.178666  1.433206   
5  8080.0  0.553946  2.770840 -5.506685 -1.059807 -0.034002  1.368250   

         max  
1  10.329649  
2  10.900495  
3   8.951761  
4  11.314717  
5  17.878930  
X describe above
(819, 819)
Corr above
